###Transform Customer Data

1. Remove records with NULL customer_id

2. Remove exact duplicate records

3. Remove duplicate records based on created_timestamp

4. CAST the columns to the correct Data Type

5. Write transformed data to the Silver schema

1. Remove records with NULL customer_id

In [0]:
%sql
select
  *
from
  gizmobox.bronze.v_customers
where
  customer_id is not null
order by
  customer_id;

2. Remove exact duplicate records

Esme aisa data bhi rh jayega jisme customer_id to same hai pr ek me mail hai..ek me nhi
esliye total records 55 aayege.

In [0]:
%sql
select distinct *
from
  gizmobox.bronze.v_customers
where
  customer_id is not null
order by
  customer_id;

In [0]:
%sql
select
  customer_id,
  MAX(created_timestamp),
  max(date_of_birth),
  max(email),
  max(member_since),
  max(telephone)
from
  gizmobox.bronze.v_customers
where
  customer_id is not null
group by
  customer_id
order by
  customer_id

3. Remove duplicate records based on created_timestamp

In [0]:
%sql
create or replace temporary view v_customer_distinct as
select distinct
  *
from
  gizmobox.bronze.v_customers
where
  customer_id is not null

In [0]:
%sql
select
  customer_id,
  MAX(created_timestamp)
from
  v_customer_distinct
group by
  customer_id

In [0]:
%sql
with cte_max as (
    select
  customer_id,
  MAX(created_timestamp) as max_created_timestamp
from
  v_customer_distinct
group by
  customer_id
)

select t.*
from
  v_customer_distinct t
join
  cte_max m
on
  t.customer_id = m.customer_id and t.created_timestamp = m.max_created_timestamp
    

4. CAST the columns to the correct Data Type


In [0]:
%sql
with cte_max as (
    select
  customer_id,
  MAX(created_timestamp) as max_created_timestamp
from
  v_customer_distinct
group by
  customer_id
)

select cast(t.created_timestamp as timestamp) as created_timestamp,
       t.customer_id,
       t.customer_name,
       cast(t.date_of_birth as date) as date_of_birth,
       t.email,
       cast(t.member_since as date) as member_since,
       t.telephone
from
  v_customer_distinct t
join
  cte_max m
on
  t.customer_id = m.customer_id and t.created_timestamp = m.max_created_timestamp
    

5. Write transformed data to the Silver schema

In [0]:
%sql
create table if not exists gizmobox.silver.customers
as
with cte_max as (
    select
  customer_id,
  MAX(created_timestamp) as max_created_timestamp
from
  v_customer_distinct
group by
  customer_id
)

select cast(t.created_timestamp as timestamp) as created_timestamp,
       t.customer_id,
       t.customer_name,
       cast(t.date_of_birth as date) as date_of_birth,
       t.email,
       cast(t.member_since as date) as member_since,
       t.telephone
from
  v_customer_distinct t
join
  cte_max m
on
  t.customer_id = m.customer_id and t.created_timestamp = m.max_created_timestamp
    

In [0]:
%sql
describe extended gizmobox.silver.customers